### Dylan Renard
### Applied Machine Learning JHU
### Homework 1
### Professor Alhassan
### Feb 1st, 2026


## HW Assignment 2: Applied Machine Learning @JHU

In [17]:
# %pip install pandas
import pandas as pd 


#### Q1 [20 pts] At a high level (i.e., without entering into mathematical details), please describe, compare, and contrast the following classifiers:

* Perceptron (textbook's version)
* SVM 
* Decision Tree
* Random Forrest (you have to research a bit about this classifier)

##### Some Comparison criteria can be:
* Speed?
* Strength?
* Robustness?
* The feature type that the classifier naturally uses (e.g. Relying on distance means that numerical features are naturally used)
* Is it statistical?
* Does the method solve an optimization problem? If yes, what is the cost function?
Which one will be the first that you would try on a dataset?





#### Q2 [20 pts] Define the following feature types and give example values from a dataset. 
#### You can pull examples from an existing dataset (like the Iris dataset) or you could write out a dataset yourself. (Hint: In order to give examples for each feature type, you will probably have to use more than one dataset.) 
* Numerical
* Nominal
* Date
* Text
* Image
* Dependent variable

#### Q3 [20 pts] Using online resources, research and find other classifier performance metrics which are also as common as the accuracy metric. Provide the mathematical equations for them and explain in your own words the meaning of the different metrics you found.

#### Note that providing mathematical equations might involve defining some more fundamentalterms, e.g. you should define “False Positive,” if you answer with a metric that builds on that.


#### Q4 [40 pts] Implement a correlation program from scratch to look at the correlations between the features of Admission_Predict.csv dataset file. (This Graduate Admission dataset, with 9 features and 500 data points, is not provided on Canvas; you have to download it from Kaggle by following the instructions in the module Jupyter notebook.) Remember, you are not allowed to used numpy functions such as mean(), stdev(), cov(), etc. You may use DataFrame.corr() only to verify the correctness of your from-scratch matrix.

In [18]:
# Loading dataset
df = pd.read_csv("Admission_Predict_Ver1.1.csv")

# Remove leading/trailing whitespace from column names
df.columns = df.columns.str.strip()

# Inspect Column Names (important for catching trailing spaces)
print(list(df.columns))

features = df.drop(columns=["Serial No."])
features.shape


['Serial No.', 'GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA', 'Research', 'Chance of Admit']


(500, 8)

In [19]:
# From-scratch statistical helper functions
#
# These functions implement the components required to compute
# Pearson correlation without using NumPy statistical utilities.
# Writing them explicitly illustrates how correlation is built
# from centering, scaling, and joint variation of features.

def mean(x):
    """Compute the arithmetic mean of a 1D list of values."""
    return sum(x) / len(x)


def std(x):
    """Compute the population standard deviation of a 1D list."""
    mu = mean(x)
    return (sum((xi - mu) ** 2 for xi in x) / len(x)) ** 0.5


def covariance(x, y):
    """Compute the population covariance between two feature vectors."""
    mx, my = mean(x), mean(y)
    return sum((xi - mx) * (yi - my) for xi, yi in zip(x, y)) / len(x)


def pearson_corr(x, y):
    """Compute the Pearson correlation coefficient between two features."""
    sx, sy = std(x), std(y)
    if sx == 0 or sy == 0:
        return 0.0
    return covariance(x, y) / (sx * sy)


# Method: compute Pearson correlation matrix from scratch
def compute_correlation_matrix(df):
    """Compute a feature-by-feature Pearson correlation matrix."""
    cols = list(df.columns)
    corr_matrix = pd.DataFrame(index=cols, columns=cols, dtype=float)

    for c1 in cols:
        x = df[c1].tolist()
        for c2 in cols:
            y = df[c2].tolist()
            corr_matrix.loc[c1, c2] = pearson_corr(x, y)

    return corr_matrix

In [20]:
# Run the from-scratch correlation matrix method
corr_scratch = compute_correlation_matrix(features)
corr_scratch

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
GRE Score,1.000000,0.827200,0.635376,0.613498,0.524679,0.825878,0.563398,0.810351
TOEFL Score,0.827200,1.000000,0.649799,0.644410,0.541563,0.810574,0.467012,0.792228
University Rating,0.635376,0.649799,1.000000,0.728024,0.608651,0.705254,0.427047,0.690132
SOP,0.613498,0.644410,0.728024,1.000000,0.663707,0.712154,0.408116,0.684137
LOR,0.524679,0.541563,0.608651,0.663707,1.000000,0.637469,0.372526,0.645365
CGPA,0.825878,0.810574,0.705254,0.712154,0.637469,1.000000,0.501311,0.882413
Research,0.563398,0.467012,0.427047,0.408116,0.372526,0.501311,1.000000,0.545871
Chance of Admit,0.810351,0.792228,0.690132,0.684137,0.645365,0.882413,0.545871,1.000000


In [21]:
# Validation (Pandas built-in correlation — verification only)
corr_pandas = features.corr()
corr_pandas

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
GRE Score,1.000000,0.827200,0.635376,0.613498,0.524679,0.825878,0.563398,0.810351
TOEFL Score,0.827200,1.000000,0.649799,0.644410,0.541563,0.810574,0.467012,0.792228
University Rating,0.635376,0.649799,1.000000,0.728024,0.608651,0.705254,0.427047,0.690132
SOP,0.613498,0.644410,0.728024,1.000000,0.663707,0.712154,0.408116,0.684137
LOR,0.524679,0.541563,0.608651,0.663707,1.000000,0.637469,0.372526,0.645365
CGPA,0.825878,0.810574,0.705254,0.712154,0.637469,1.000000,0.501311,0.882413
Research,0.563398,0.467012,0.427047,0.408116,0.372526,0.501311,1.000000,0.545871
Chance of Admit,0.810351,0.792228,0.690132,0.684137,0.645365,0.882413,0.545871,1.000000


In [23]:
# Quantitative validation: maximum absolute difference
max_abs_diff = (corr_scratch - corr_pandas).abs().to_numpy().max()
print(max_abs_diff, type(max_abs_diff), " Dang that is a small error sheeeeeeesh! :-O")

1.2212453270876722e-15 <class 'numpy.float64'>  Dang that is a small error sheeeeeeesh! :-O


#### References:

Datta, N. (2020, October 25). 📖 graduate admission chances 📕 📔. Kaggle. https://www.kaggle.com/code/nitindatta/graduate-admission-chances/input 

Raschka, S., Liu, Y., & Mirjalili, V. (2022). Machine learning with pytorch and scikit-learn : develop machine learning and deep learning models with python. Packt Publishing.

